In [1]:
import os
import copernicusmarine

# Locally: interactive. On GCP: set these env vars from Secret Manager.
# COPERNICUSMARINE_SERVICE_USERNAME / COPERNICUSMARINE_SERVICE_PASSWORD
copernicusmarine.login()  # prompts once, stores creds locally

/Users/claytonpreston/Documents/NextWave/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
INFO - 2026-11-16T21:01:54Z - Using existing credentials from /Users/claytonpreston/.copernicusmarine/.copernicusmarine-credentials. Use --force-overwrite combined with credentials (specified by arguments, netrc file or environment variables) to always overwrite.


True

In [2]:
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import xarray as xr
import requests

TZ = "Europe/Athens"
# GREECE
# BBOX = dict(minimum_longitude=19.0, maximum_longitude=30.0, minimum_latitude=34.0, maximum_latitude=42.0)
# PELOPO
BBOX = dict(minimum_longitude=20.5, maximum_longitude=23.7, minimum_latitude=36.0, maximum_latitude=38.5)

# facing = compass direction the beach looks toward; window = half-width of good swell angle
SPOTS = {}

now = datetime.now(timezone.utc).replace(minute=0, second=0, microsecond=0, tzinfo=None)
START, END = now - timedelta(hours=6), now + timedelta(days=7)

## spots

In [3]:
SPOTS["lagouvardos"] = dict(
    name="Lagouvardos",
    region="Peloponnese",          # Messinia, west coast

    # Location (beach itself; wave_point() snaps to the nearest sea cell)
    lat=37.09,
    lon=21.58,

    # Geometry, all "coming from" degrees
    facing=260,                    # beach looks WSW/W (map shows a N-S coast, sea to the west)
    window=60,                     # swell window 200-320, covers SSW through NW
    ideal_swell_dir=(235, 285),    # WSW to W is the sweet spot

    # Size and period (starting guesses, tune from real sessions)
    min_hs=1.0,                    # below this, "not surfable"
    max_hs=2.5,                    # above this, closes out / too much
    min_period=5.0,                # short-fetch Med chop floor
    ideal_period=(6.0, 9.0),

    # Wind, relative to facing 260 (offshore blows from the east)
    offshore_dir=(40, 130),        # clean
    max_onshore_kn=8,              # beyond this, wind waves dominate and it gets messy
    max_offshore_kn=20,            # beyond this, it blows the face out flat

    # Type
    break_type="beach",
    notes="Exposed beach break, works occasionally. Best on W/WSW swell with E wind.",
)

# SPOTS["vouliagmeni"] = dict(
#     name="Vouliagmeni",
#     region="Attica",               # Saronic Gulf, Athens Riviera

#     # Location (approximate, confirm against your pin; wave_point() snaps to nearest sea cell)
#     lat=37.81,
#     lon=23.78,

#     # Geometry, all "coming from" degrees
#     facing=180,                    # bay mouth opens due south (map shows the cone pointing S)
#     window=40,                     # swell window 140-220, narrow because the bay is sheltered
#     ideal_swell_dir=(165, 205),    # S to SSW

#     # Size and period (starting guesses, tune from real sessions)
#     min_hs=0.4,                    # sheltered bay, so smaller waves count
#     max_hs=1.5,
#     min_period=3.5,                # Saronic events are short-period wind waves
#     ideal_period=(4.0, 7.0),

#     # Wind, relative to facing 180 (offshore blows from the north)
#     offshore_dir=(320, 40),        # wraps through 0, so use angdiff(), not a plain range check
#     max_onshore_kn=15,             # south wind makes the waves here, but too much turns it to mush
#     max_offshore_kn=20,

#     # Swell source
#     use_wind_waves=True,           # judge on total Hs / VHM0_WW, not only the SW1 swell partition

#     break_type="beach/bay",
#     notes="Saronic Gulf: only works on S/SSW wind events (notos). Short period, wind dependent.",
# )

# SPOTS["messakti"] = dict(
#     name="Messakti",
#     region="Cyclades/East Aegean",   # Ikaria, north coast

#     # Location (approximate, confirm against your pin; wave_point() snaps to nearest sea cell)
#     lat=37.63,
#     lon=26.03,

#     # Geometry, all "coming from" degrees
#     facing=350,                      # bay opens north (cone points NNW in the map)
#     window=50,                       # swell window 300-40, wraps through 0
#     ideal_swell_dir=(340, 30),       # NNW to NNE, meltemi sector

#     # Size and period (starting guesses, tune from real sessions)
#     min_hs=0.7,
#     max_hs=3.0,
#     min_period=4.0,                  # meltemi wind sea is short and steep
#     ideal_period=(5.0, 8.0),

#     # Wind, relative to facing 350 (offshore blows from the south)
#     offshore_dir=(140, 220),
#     max_onshore_kn=30,               # meltemi IS onshore here, so the cap is high
#     max_offshore_kn=20,

#     # Meltemi driver
#     use_wind_waves=True,             # judge on total Hs / VHM0_WW, not the swell partition
#     driver_wind_dir=(340, 50),       # N to NE
#     min_driver_wind_kn=15,           # needs a real meltemi, not just a breeze
#     season_months=(6, 9),            # Jun-Sep; peaks Jul-Aug

#     break_type="beach/bay",
#     notes="Works almost only on summer meltemi wind-sea. Wind and waves arrive together.",
# )

## Wave data
https://data.marine.copernicus.eu/product/MEDSEA_ANALYSISFORECAST_WAV_006_017/description

Coordinate reference system
WGS 84 (EPSG:4326)
Data assimilation
SWHWind speed
Update frequency
Daily – 06:00; 20:00
Format
NetCDF-4
Originating centre
HCMR (Greece)

In [4]:
cat = copernicusmarine.describe(product_id="MEDSEA_ANALYSISFORECAST_WAV_006_017")
for p in cat.products:
    for d in p.datasets:
        print(d.dataset_id)

# cmems_mod_med_wav_anfc_4.2km_PT1H-i
# cmems_mod_med_wav_anfc_4.2km_static
WAVE_DATASET_ID = "cmems_mod_med_wav_anfc_4.2km_PT1H-i"

Fetching catalogue 1: 100%|██████████| 2/2 [00:00<00:00,  2.30it/s]

cmems_mod_med_wav_anfc_4.2km_PT1H-i
cmems_mod_med_wav_anfc_4.2km_static


### get datasets

In [5]:
WANTED = ["VHM0", "VTPK", "VTM10", "VMDR",              # total sea state
          "VHM0_WW", "VTM01_WW", "VMDR_WW",             # local wind waves
          "VHM0_SW1", "VTM01_SW1", "VMDR_SW1"]          # primary swell
WAVE_VARS = [v for v in WANTED if v in ds.data_vars]

In [7]:
# Download once to a local NetCDF. Tight bbox around your spots keeps it small.
copernicusmarine.subset(
    dataset_id=WAVE_DATASET_ID,
    start_datetime=START.strftime("%Y-%m-%dT%H:%M:%S"),
    end_datetime=END.strftime("%Y-%m-%dT%H:%M:%S"),
    output_filename="waves.nc",
    output_directory="data",
    overwrite=True,
    **BBOX,
)

# local in-mem file downloaded data
ds = xr.open_dataset("data/waves.nc").load()

INFO - 2026-11-16T21:02:20Z - Selected dataset version: "202311"
INFO - 2026-11-16T21:02:20Z - Selected dataset part: "default"
100%|██████████| [06:51<00:00]  
INFO - 2026-11-16T21:09:13Z - Total size of the download: 29.33 MB.


In [8]:
# Extract a point (snaps to nearest sea cell, since coastal cells can be NaN)
def nearest_ocean_cell(da2d, lat, lon):
    lats, lons = np.meshgrid(da2d.latitude.values, da2d.longitude.values, indexing="ij")
    dist = np.hypot(lats - lat, (lons - lon) * np.cos(np.radians(lat)))
    dist = np.where(np.isnan(da2d.values), np.inf, dist)
    return np.unravel_index(dist.argmin(), dist.shape)

def wave_point(ds, lat, lon, variables):
    first = ds["VHM0"].isel(time=0).load()
    i, j = nearest_ocean_cell(first, lat, lon)
    df = ds[variables].isel(latitude=i, longitude=j).to_dataframe()
    df = df.drop(columns=[c for c in ("latitude", "longitude") if c in df.columns])
    df.index = pd.to_datetime(df.index).tz_localize("UTC").tz_convert(TZ)
    df.index.name = "time"
    return df


# Wind from Open-Meteo
def get_wind(lat, lon, days=7, models=None):
    params = dict(
        latitude=lat, longitude=lon,
        hourly="wind_speed_10m,wind_direction_10m,wind_gusts_10m",
        wind_speed_unit="kn", timezone="UTC", forecast_days=days,
    )
    if models:  # e.g. "icon_d2", "ecmwf_ifs025", "gfs_seamless"
        params["models"] = models
    r = requests.get("https://api.open-meteo.com/v1/forecast", params=params, timeout=30)
    r.raise_for_status()
    df = pd.DataFrame(r.json()["hourly"])
    df["time"] = pd.to_datetime(df["time"], utc=True)
    return (df.set_index("time").tz_convert(TZ)
              .rename(columns={"wind_speed_10m": "wind_kn",
                               "wind_direction_10m": "wind_dir",
                               "wind_gusts_10m": "gust_kn"}))

# Merge and derive per-spot fields
def angdiff(a, b):
    """Signed smallest angle a-b in degrees, [-180, 180)."""
    return (a - b + 180) % 360 - 180

def build_frame(waves, wind, spot):
    df = waves.join(wind, how="inner")
    df["swell_off"] = angdiff(df["VMDR"], spot["facing"]).abs()
    df["in_window"] = df["swell_off"] <= spot["window"]

    w_off = angdiff(df["wind_dir"], spot["facing"]).abs()
    df["wind_rel"] = np.where(w_off >= 120, "offshore",
                      np.where(w_off <= 60, "onshore", "cross"))

    tm = df["VTM10"] if "VTM10" in df else df["VTPK"]
    df["power_kw_m"] = 0.49 * df["VHM0"] ** 2 * tm     # deep-water power, relative use only
    df["swell_share"] = df.get("VHM0_SW1", np.nan) / df["VHM0"]  # swell vs wind-wave mix
    return df

In [11]:
def in_dataset(ds, lat, lon, margin=0.1):
    """True if the point lies inside the downloaded box (with a small margin)."""
    return (ds.latitude.min() + margin <= lat <= ds.latitude.max() - margin and
            ds.longitude.min() + margin <= lon <= ds.longitude.max() - margin)

frames = {}
for name, s in SPOTS.items():
    if not in_dataset(ds, s["lat"], s["lon"]):
        print(f"skip {name}: outside the downloaded box")
        continue
    w = wave_point(ds, s["lat"], s["lon"], WAVE_VARS)
    wi = get_wind(s["lat"], s["lon"])
    frames[name] = build_frame(w, wi, s)

frames["lagouvardos"][["VHM0", "VTPK", "VMDR", "wind_kn", "wind_rel", "in_window", "power_kw_m"]].round(1)

NameError: name 'WAVE_VARS' is not defined

In [ ]:
import json

# spots (tuples become JSON arrays automatically)
json.dump(SPOTS, open("spots.json", "w"))

# forecasts: drop NaN rows, one list of hourly records per spot
need = ["VHM0", "VTPK", "VMDR", "wind_kn", "wind_dir"]
forecast = {
    name: json.loads(df.dropna(subset=need).reset_index().to_json(orient="records", date_format="iso"))
    for name, df in frames.items()
}
json.dump(forecast, open("forecast.json", "w"))